# 03 — Fine-tuning BLIP on a VQAv2 subset

**Goal:** actually fine-tune the pretrained BLIP VQA model on the subset
prepared in notebook 02, and save a real checkpoint to `models/visionquery-finetuned/`.

**Run this on a Colab T4 GPU.** With the default config (subset size ~3000,
batch size 4, grad accumulation 4, 3 epochs) this should take on the order of
tens of minutes on a T4 — actual time will be printed at the end and should be
recorded in `EXPERIMENT_LOG.md`, not guessed.

In [ ]:
!pip install -q transformers datasets torch pyyaml accelerate

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)  # so relative paths in config.yaml (./models/...) resolve correctly

from src.train import load_config, run_training
cfg = load_config("configs/config.yaml")
cfg["training"]  # inspect hyperparameters before running - edit configs/config.yaml to change them

In [ ]:
import time
start = time.time()
log_rows = run_training(cfg)
elapsed_minutes = (time.time() - start) / 60
print(f"Training took {elapsed_minutes:.1f} minutes")
# Record elapsed_minutes, and the log_rows losses, in EXPERIMENT_LOG.md

**Expected output:** one printed line per epoch, e.g.
`Epoch 1/3 - train_loss=X.XXXX - val_loss=X.XXXX`, followed by
`Saved model + processor to: ./models/visionquery-finetuned` and the total
training time. **These numbers are only meaningful once you've actually run
this cell** — do not paste example numbers into your README as if they were real.

**Common errors & fixes:**
- `CUDA out of memory` → lower `training.batch_size` in `configs/config.yaml` (e.g. to 2), or increase `gradient_accumulation_steps` to keep the effective batch size similar.
- Training looks very slow → confirm the Colab runtime is actually GPU (see notebook 01), and that `training.fp16: true` in the config.
- Loss is `nan` → lower `learning_rate` (e.g. to 2e-5) and re-run.


In [ ]:
import pandas as pd
pd.DataFrame(log_rows)

**Next notebook:** `04_evaluation.ipynb` — this loads the checkpoint saved above and measures real VQA accuracy.